# core

> Fill in a module description here

In [ ]:
#| default_exp core

In [ ]:
#| export
import inspect,json,os,importlib.util
from fastcore.script import call_parse
from fastcore.utils import *
from shutil import copytree
from importlib.resources import files, as_file

In [ ]:
#| export
_settings_tmpl = '''from fastcore.utils import AttrDict

cfg = AttrDict(
    name={name!r},
    port={port!r},
    email={email!r},
    description={description!r},
    github={github!r},
    twitter={twitter!r},
    linkedin={linkedin!r},
)
'''

@call_parse
def solveblog_new(
    name:str='Your Name', # Your name
    port:int=8000,        # Port to serve on
    email:str='',         # Your email (optional)
    description:str='',   # RSS feed description (optional)
    github:str='',        # GitHub handle (optional)
    twitter:str='',       # Twitter/X handle (optional)
    linkedin:str='',      # LinkedIn handle (optional)
):
    "Create a new solveblog_settings.py file. Also creates demo public/ directory and custom.css file if they do not yet exist."
    Path('solveblog_settings.py').write_text(_settings_tmpl.format(**locals()))
    with as_file(files('solveblog') / 'data' / 'public') as p: copytree(p, 'public') if not Path('public').exists() else None
    if not Path('custom.css').exists(): Path('custom.css').write_text('/* Override solveblog styles here.\n   Tip: use your browser\'s DevTools (F12) to inspect elements and find class names. */\n')
    print(f"✓ solveblog project created! Next step run: `!solveblog` to start your blog.")

In [ ]:
#| export
def load_cfg():
    "Load solveblog config from solveblog_settings.py in the current working directory."
    s = importlib.util.spec_from_file_location("solveblog_settings", os.path.join(os.getcwd(), 'solveblog_settings.py'))
    m = importlib.util.module_from_spec(s)
    s.loader.exec_module(m)
    return m.cfg

In [ ]:
#| export
@call_parse
def solveblog_run(
    reload:bool=True, # Auto-reload on changes
):
    "Run your solveblog"
    from solveblog.app import app
    from fasthtml.common import serve
    cfg = load_cfg()
    url = json.loads(os.environ['PUBLIC_DOMAINS'])[str(cfg.port)]
    print(f"\n\033[1m{'':═^78}\033[0m")
    print(f"\033[1m{'>>> Serving your SolveIt blog at: ' + url + '  ':^60}\033[0m")
    print(f"\033[1m{'':═^78}\033[0m\n")
    serve(appname='solveblog.app', port=int(cfg.port), reload=reload, reload_dirs=['.'],
      reload_includes=["*.py", "*.ipynb", "*.md", "*.png", "*.jpg", "*.jpeg", "*.gif", "*.svg", "*.webp", "*.css"])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()